## Example 1: plotly-geo shapefile data

In [ ]:
# plotly-geo installs the `_plotly_geo` package: US state and county shapefiles that
# plotly's figure factory `create_choropleth` reads (plotly 7 removed that factory).
import importlib.metadata
import importlib.resources
import importlib.util
import warnings

import _plotly_geo
import plotly

print("Example 1: plotly-geo shapefile data")
print(f"plotly-geo {importlib.metadata.version('plotly-geo')}, plotly {plotly.__version__}")

data_dir = importlib.resources.files("_plotly_geo") / "package_data"
names = {entry.name for entry in data_dir.iterdir()}
shapefiles = sorted(name for name in names if name.endswith(".shp"))
print(f"Shapefiles: {shapefiles}")
assert any("county" in name for name in shapefiles), shapefiles

for shp in shapefiles:
    stem = shp[: -len(".shp")]
    # A shapefile is a set: geometry (.shp), index (.shx) and attributes (.dbf).
    assert {stem + ".shx", stem + ".dbf"} <= names, stem
    # Every .shp starts with the big-endian file code 9994.
    with (data_dir / shp).open("rb") as f:
        assert int.from_bytes(f.read(4), "big") == 9994, shp

# The county map needs geopandas, shapely and pyshp (imported as `shapefile`), and a plotly
# release that still has create_choropleth.
missing = [m for m in ("geopandas", "shapely", "shapefile") if importlib.util.find_spec(m) is None]
create_choropleth = None
if not missing:
    import plotly.figure_factory as ff

    create_choropleth = getattr(ff, "create_choropleth", None)

if missing:
    print(f"Skipping the county choropleth: {missing} not installed")
elif create_choropleth is None:
    print(f"Skipping the county choropleth: plotly {plotly.__version__} has no create_choropleth")
else:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # older plotly releases use deprecated shapely attributes here
        fig = create_choropleth(fips=["06037", "17031", "36061"], values=[10, 20, 30])
    assert len(fig.data) > 0, fig.data
    print(f"County choropleth built from the plotly-geo shapefiles: {len(fig.data)} traces")

print("Example 1 completed")

## Example 2: Basic scatter geo

In [ ]:
# The geo traces from here on (Scattergeo, Choropleth with ISO-3 codes, Scattermap) need only
# plotly: plotly.js draws its own base maps and does not read plotly-geo's shapefiles.

import plotly.graph_objects as go

# Example 2: Basic scatter geo plot
print("Example 2: Basic scatter geo")

cities = {
    'city': ['NYC', 'London', 'Tokyo', 'Sydney'],
    'lat': [40.71, 51.51, 35.68, -33.87],
    'lon': [-74.01, -0.13, 139.65, 151.21],
}

fig = go.Figure(go.Scattergeo(
    lon=cities['lon'],
    lat=cities['lat'],
    text=cities['city'],
    mode='markers+text'
))

fig.update_layout(geo=dict(projection_type='natural earth'))

trace = fig.data[0]
assert trace.type == 'scattergeo'
assert list(trace.lat) == cities['lat'] and list(trace.lon) == cities['lon']
assert list(trace.text) == cities['city']
assert trace.mode == 'markers+text'
assert fig.layout.geo.projection.type == 'natural earth'

print(f"Created plot with {len(trace.lat)} cities")
print("Example 2 completed")

## Example 3: Choropleth map

In [ ]:
import plotly.graph_objects as go

# Example 3: Choropleth map
print("Example 3: Choropleth map")

countries = {
    'iso': ['USA', 'CHN', 'JPN', 'DEU', 'GBR'],
    'values': [21.4, 14.3, 5.1, 3.9, 2.8]
}

fig = go.Figure(go.Choropleth(
    locations=countries['iso'],
    z=countries['values'],
    colorscale='blues',
    colorbar_title='GDP (T USD)'
))

fig.update_layout(
    title='GDP by Country',
    geo=dict(projection_type='equirectangular')
)

trace = fig.data[0]
assert trace.type == 'choropleth'
assert list(trace.locations) == countries['iso']
assert list(trace.z) == countries['values']
# The named colorscale is expanded into (position, color) pairs from 0 to 1
assert trace.colorscale[0][0] == 0 and trace.colorscale[-1][0] == 1
assert trace.colorbar.title.text == 'GDP (T USD)'
assert fig.layout.title.text == 'GDP by Country'
assert fig.layout.geo.projection.type == 'equirectangular'

print(f"Created choropleth for {len(trace.locations)} countries")
print("Example 3 completed")

## Example 4: Map projections

In [ ]:
import plotly.graph_objects as go

# Example 4: Map projections
print("Example 4: Map projections")

locs = {'lat': [0, 30, -30], 'lon': [0, 60, -60]}

projections = ['orthographic', 'natural earth', 'mercator']

for proj in projections:
    fig = go.Figure(go.Scattergeo(
        lon=locs['lon'],
        lat=locs['lat'],
        mode='markers'
    ))
    fig.update_layout(geo=dict(projection_type=proj))
    assert fig.layout.geo.projection.type == proj
    assert list(fig.data[0].lat) == locs['lat']
    print(f"Created {proj} projection")

# Test specific projection
fig_ortho = go.Figure(go.Scattergeo(lon=[0], lat=[0], mode='markers'))
fig_ortho.update_layout(geo=dict(projection_type='orthographic', projection_rotation=dict(lon=30, lat=10)))
assert fig_ortho.layout.geo.projection.type == 'orthographic'
assert (fig_ortho.layout.geo.projection.rotation.lon, fig_ortho.layout.geo.projection.rotation.lat) == (30, 10)

# An unknown projection name is rejected by plotly's property validation
try:
    go.Figure().update_layout(geo=dict(projection_type='not-a-projection'))
except ValueError:
    print("Unknown projection rejected")
else:
    raise AssertionError("an unknown projection_type should raise ValueError")

print("Example 4 completed")

## Example 5: Geographic lines

In [ ]:
import plotly.graph_objects as go

# Example 5: Geographic lines
print("Example 5: Geographic lines")

# Flight path
path = {
    'cities': ['NYC', 'Chicago', 'LA'],
    'lat': [40.71, 41.88, 34.05],
    'lon': [-74.01, -87.63, -118.24]
}

fig = go.Figure()

# Add line
fig.add_trace(go.Scattergeo(
    lon=path['lon'],
    lat=path['lat'],
    mode='lines+markers',
    line=dict(width=2, color='red'),
    marker=dict(size=8)
))

fig.update_layout(
    title='Flight Path',
    geo=dict(projection_type='albers usa')
)

trace = fig.data[0]
assert len(fig.data) == 1 and trace.type == 'scattergeo'
assert trace.mode == 'lines+markers'
assert list(trace.lat) == path['lat'] and list(trace.lon) == path['lon']
assert trace.line.width == 2 and trace.line.color == 'red'
assert trace.marker.size == 8
assert fig.layout.title.text == 'Flight Path'
assert fig.layout.geo.projection.type == 'albers usa'

print(f"Created path with {len(trace.lat)} cities")
print("Example 5 completed")

## Example 6: Scattermap

In [ ]:
import plotly.graph_objects as go

print("Example 6: Scattermap")

assert hasattr(go, "Scattermap"), "go.Scattermap missing -- the MapLibre replacement trace is gone"

cities = {
    'lat': [40.71, 34.05, 41.88],
    'lon': [-74.01, -118.24, -87.63],
    'names': ['NYC', 'LA', 'Chicago']
}

fig = go.Figure(go.Scattermap(
    lat=cities['lat'],
    lon=cities['lon'],
    text=cities['names'],
    mode='markers'
))

fig.update_layout(
    map=dict(
        style='open-street-map',
        center=dict(lat=39, lon=-95),
        zoom=3
    )
)

# The round-trip is the real assertion: a renamed trace that does not survive `to_plotly_json`
# is a trace plotly cannot render.
spec = fig.to_plotly_json()
assert spec["data"][0]["type"] == "scattermap", spec["data"][0]["type"]
assert spec["layout"]["map"]["zoom"] == 3, spec["layout"]["map"]
assert len(spec["data"][0]["lat"]) == 3, spec["data"][0]["lat"]

print(f"Created map plot with {len(cities['lat'])} points")
print("Example 6 completed")